<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/GES3_00_Archive_Manifest_and_Schema_Audit_COLAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GES 3.0 — Notebook 00
## ClinVar Archive Manifest & XML Schema Audit

**Google Colab / GitHub-ready**

This is the first new GES 3.0 notebook. It implements **Stage 00: `archive_manifest_and_schema_audit`** from the high-complexity project blueprint.

### Scientific purpose
GES 3.0 requires a release-aware, leakage-controlled, multi-snapshot ClinVar history before any forecasting model is built. This notebook therefore **does not train a model** and **does not bulk-download multi-gigabyte XML releases by default**.

It creates a reproducible inventory of monthly ClinVar XML releases and the schema generations that a later parser must bridge.

### Outputs
The notebook writes:

- `clinvar_release_manifest_all.csv`
- `clinvar_release_manifest_canonical.csv`
- `clinvar_month_coverage.csv`
- `clinvar_schema_catalog.csv`
- `clinvar_schema_bridge_plan.csv`
- `stage00_audit_summary.json`
- `stage00_runtime_metadata.json`
- `GES3_STAGE00_ARTIFACTS.zip`

### Key design rules
1. Preserve the existing GES repository as the locked baseline.
2. Inventory **monthly archived releases**, not weekly transient releases.
3. Keep VCV and RCV data models distinct.
4. Prefer the **current XML format** where the same month exists in both current and legacy formats.
5. Retain legacy releases where needed to cover the candidate 5–6 year history.
6. Record NCBI-provided checksums and source lineage.
7. Do not mix germline and somatic classification axes in downstream modeling.
8. Do not freeze the final study window in this notebook; first audit archive coverage and schema transitions.

### Primary NCBI references
- ClinVar downloads: https://www.ncbi.nlm.nih.gov/clinvar/docs/downloads/
- ClinVar release cycle: https://www.ncbi.nlm.nih.gov/clinvar/docs/release_cycle/
- ClinVar FTP primer: https://www.ncbi.nlm.nih.gov/clinvar/docs/ftp_primer/
- XML README: https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/_README

## 1. Install lightweight dependencies

Colab already contains most scientific Python packages. We only ensure that the HTML/XML and Parquet dependencies used by this audit are present.

In [1]:
!pip -q install beautifulsoup4 lxml pyarrow tqdm

## 2. Imports, runtime metadata, and configuration

The default candidate history begins in **2021**, giving approximately a 5–6 year window by 2026. This is **not the final frozen cohort window**. The exact window should be frozen only after this archive/schema audit and the later event-prevalence analysis.

In [2]:
from __future__ import annotations

import concurrent.futures as cf
import hashlib
import json
import os
import platform
import re
import sys
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urljoin

import bs4
import lxml
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import display
from tqdm.auto import tqdm

START_YEAR = 2021
END_YEAR = None
INVENTORY_BOTH_MODELS = True
REQUEST_TIMEOUT = 60
MAX_WORKERS = 8

# Safety: Stage 00 never bulk-downloads the multi-GB monthly XML archives.
ALLOW_FULL_XML_DOWNLOAD = False

OUTDIR = Path("/content/ges3_stage00")
OUTDIR.mkdir(parents=True, exist_ok=True)

RUN_UTC = datetime.now(timezone.utc).isoformat()

print("Stage 00 started:", RUN_UTC)
print("Output directory:", OUTDIR)
print("Candidate start year:", START_YEAR)

Stage 00 started: 2026-08-16T17:11:48.502727+00:00
Output directory: /content/ges3_stage00
Candidate start year: 2021


## 3. Source registry

NCBI provides separate VCV and RCV XML products. SCV submitted assertions are nested in these records. GES 3.0 will use the **RCV-month** as its primary prediction unit while retaining VCV information for variant identity and supporting structure.

The current XML format separates classification types such as germline and somatic classifications; the older XML format used a single classification representation. The schema bridge built later must therefore normalize semantics explicitly rather than treating all historical XML as identical.

In [3]:
BASE = "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/"

SOURCE_REGISTRY = pd.DataFrame([
    {
        "data_model": "VCV",
        "format_generation": "current",
        "root_url": BASE,
        "archive_url": BASE + "archive/",
        "filename_prefix": "ClinVarVCVRelease_",
        "supported_status": "current",
    },
    {
        "data_model": "RCV",
        "format_generation": "current",
        "root_url": BASE + "RCV_release/",
        "archive_url": BASE + "RCV_release/archive/",
        "filename_prefix": "ClinVarRCVRelease_",
        "supported_status": "current",
    },
    {
        "data_model": "VCV",
        "format_generation": "legacy",
        "root_url": BASE + "VCV_xml_old_format/",
        "archive_url": BASE + "VCV_xml_old_format/archive/",
        "filename_prefix": "ClinVarVariationRelease_",
        "supported_status": "historical_legacy",
    },
    {
        "data_model": "RCV",
        "format_generation": "legacy",
        "root_url": BASE + "RCV_xml_old_format/",
        "archive_url": BASE + "RCV_xml_old_format/archive/",
        "filename_prefix": "ClinVarFullRelease_",
        "supported_status": "historical_legacy",
    },
])

display(SOURCE_REGISTRY)

,data_model,format_generation,root_url,archive_url,filename_prefix,supported_status
0,VCV,current,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/a...,ClinVarVCVRelease_,current
1,RCV,current,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,ClinVarRCVRelease_,current
2,VCV,legacy,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...,ClinVarVariationRelease_,historical_legacy
3,RCV,legacy,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,ClinVarFullRelease_,historical_legacy


## 4. HTTP and Apache-index helpers

These functions read directory listings, HTTP headers, tiny `.md5` files, XSD listings, and the small official sample XML. They do **not** download the full monthly XML archives.

In [4]:
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "GES3-Archive-Audit/1.0 (research; ClinVar public FTP via HTTPS)"
})

def get_text(url: str, timeout: int = REQUEST_TIMEOUT) -> str:
    r = SESSION.get(url, timeout=timeout)
    r.raise_for_status()
    return r.text

def list_apache_links(url: str) -> pd.DataFrame:
    """Parse an Apache-style directory index into href/name/url rows."""
    html = get_text(url)
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        name = a.get_text(" ", strip=True) or href
        if href in ("../", "/"):
            continue
        rows.append({
            "parent_url": url,
            "name": name,
            "href": href,
            "url": urljoin(url, href),
        })
    if not rows:
        return pd.DataFrame(columns=["parent_url", "name", "href", "url"])
    return pd.DataFrame(rows).drop_duplicates(subset=["url"]).reset_index(drop=True)

def safe_head(url: str) -> dict:
    try:
        r = SESSION.head(url, allow_redirects=True, timeout=REQUEST_TIMEOUT)
        if r.status_code >= 400:
            r = SESSION.get(url, stream=True, timeout=REQUEST_TIMEOUT)
        h = r.headers
        content_length = h.get("Content-Length")
        return {
            "http_status": int(r.status_code),
            "content_length": int(content_length) if str(content_length).isdigit() else None,
            "last_modified": h.get("Last-Modified"),
            "etag": h.get("ETag"),
            "accept_ranges": h.get("Accept-Ranges"),
            "head_error": None,
        }
    except Exception as e:
        return {
            "http_status": None,
            "content_length": None,
            "last_modified": None,
            "etag": None,
            "accept_ranges": None,
            "head_error": repr(e),
        }

def fetch_md5(md5_url: str) -> tuple[str | None, str | None]:
    """Fetch NCBI's tiny companion .md5 file; return checksum and raw text."""
    try:
        text = get_text(md5_url).strip()
        m = re.search(r"\b([a-fA-F0-9]{32})\b", text)
        return (m.group(1).lower() if m else None, text)
    except Exception as e:
        return (None, f"ERROR: {e!r}")

def parse_release_month(filename: str) -> str | None:
    m = re.search(r"_(\d{4})-(\d{2})\.xml\.gz$", filename)
    if not m:
        return None
    return f"{m.group(1)}-{m.group(2)}"

def is_monthly_release_file(name: str, prefix: str) -> bool:
    pattern = re.escape(prefix) + r"\d{4}-\d{2}\.xml\.gz"
    return bool(re.fullmatch(pattern, name))

def discover_archive_years(archive_url: str) -> list[int]:
    try:
        idx = list_apache_links(archive_url)
    except Exception:
        return []
    years = []
    for href in idx.get("href", []):
        m = re.fullmatch(r"(\d{4})/", str(href))
        if m:
            years.append(int(m.group(1)))
    return sorted(set(years))

print("Helper functions loaded.")

Helper functions loaded.


## 5. Discover monthly releases

Discovery is **manifest-driven**. It scans the current NCBI directory structure and year-specific archives, then keeps only monthly comprehensive files matching the official filename patterns. `00-latest` symlinks and weekly releases are excluded.

In [5]:
def discover_source_monthlies(row: pd.Series) -> pd.DataFrame:
    data_model = row["data_model"]
    fmt = row["format_generation"]
    root_url = row["root_url"]
    archive_url = row["archive_url"]
    prefix = row["filename_prefix"]

    frames = []

    try:
        root_idx = list_apache_links(root_url)
        if not root_idx.empty:
            root_idx["source_scope"] = "root"
            frames.append(root_idx)
    except Exception as e:
        print(f"[WARN] Root listing failed: {root_url}: {e!r}")

    archive_years = discover_archive_years(archive_url)
    for year in archive_years:
        if year < START_YEAR:
            continue
        if END_YEAR is not None and year > END_YEAR:
            continue

        year_url = urljoin(archive_url, f"{year}/")
        try:
            y = list_apache_links(year_url)
            if not y.empty:
                y["source_scope"] = f"archive/{year}"
                frames.append(y)
        except Exception as e:
            print(f"[WARN] Archive listing failed: {year_url}: {e!r}")

    if not frames:
        return pd.DataFrame()

    d = pd.concat(frames, ignore_index=True)
    d = d[d["name"].map(lambda x: is_monthly_release_file(str(x), prefix))].copy()

    d["data_model"] = data_model
    d["format_generation"] = fmt
    d["supported_status"] = row["supported_status"]
    d["release_month"] = d["name"].map(parse_release_month)
    d["release_period"] = pd.PeriodIndex(d["release_month"], freq="M")
    d["year"] = d["release_period"].dt.year
    d["month"] = d["release_period"].dt.month
    d["md5_url"] = d["url"] + ".md5"

    d = d[d["year"] >= START_YEAR]
    if END_YEAR is not None:
        d = d[d["year"] <= END_YEAR]

    return d.reset_index(drop=True)

release_parts = []

for _, src in SOURCE_REGISTRY.iterrows():
    if (not INVENTORY_BOTH_MODELS) and src["data_model"] != "RCV":
        continue
    print(f"Discovering {src['data_model']} / {src['format_generation']} ...")
    part = discover_source_monthlies(src)
    print("  releases found:", len(part))
    release_parts.append(part)

manifest = pd.concat(release_parts, ignore_index=True) if release_parts else pd.DataFrame()

if manifest.empty:
    raise RuntimeError(
        "No ClinVar monthly releases were discovered. "
        "Check NCBI connectivity or source paths."
    )

manifest = (
    manifest
    .drop_duplicates(subset=["data_model", "format_generation", "release_month", "url"])
    .sort_values(["release_period", "data_model", "format_generation"])
    .reset_index(drop=True)
)

print("Total discovered monthly XML files:", len(manifest))
display(manifest[[
    "release_month", "data_model", "format_generation",
    "source_scope", "name", "url"
]].head(20))

Discovering VCV / current ...
  releases found: 31
Discovering RCV / current ...
  releases found: 31
Discovering VCV / legacy ...
  releases found: 55
Discovering RCV / legacy ...
  releases found: 55
Total discovered monthly XML files: 172


,release_month,data_model,format_generation,source_scope,name,url
0,2021-01,RCV,legacy,archive/2021,ClinVarFullRelease_2021-01.xml.gz,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...
1,2021-01,VCV,legacy,archive/2021,ClinVarVariationRelease_2021-01.xml.gz,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...
2,2021-02,RCV,legacy,archive/2021,ClinVarFullRelease_2021-02.xml.gz,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...
3,2021-02,VCV,legacy,archive/2021,ClinVarVariationRelease_2021-02.xml.gz,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...
4,2021-03,RCV,legacy,archive/2021,ClinVarFullRelease_2021-03.xml.gz,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...
5,2021-03,VCV,legacy,archive/2021,ClinVarVariationRelease_2021-03.xml.gz,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...
6,2021-04,RCV,legacy,archive/2021,ClinVarFullRelease_2021-04.xml.gz,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...
7,2021-04,VCV,legacy,archive/2021,ClinVarVariationRelease_2021-04.xml.gz,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...
8,2021-05,RCV,legacy,archive/2021,ClinVarFullRelease_2021-05.xml.gz,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...
9,2021-05,VCV,legacy,archive/2021,ClinVarVariationRelease_2021-05.xml.gz,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...


## 6. Attach remote file metadata and NCBI-provided MD5 checksums

Monthly ClinVar XML files are multiple gigabytes, so Stage 00 does **not** hash the full remote file locally. Instead it records NCBI's published MD5 companion checksum and remote metadata.

Stage 01 must verify the MD5 after each actual download and compute a local SHA-256 for immutable source and normalized artifacts.

In [6]:
def enrich_one(rec: dict) -> dict:
    head = safe_head(rec["url"])
    md5_hex, md5_raw = fetch_md5(rec["md5_url"])
    return {
        **rec,
        **head,
        "ncbi_md5": md5_hex,
        "ncbi_md5_raw": md5_raw,
    }

records = manifest.to_dict("records")
enriched = []

with cf.ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = [ex.submit(enrich_one, r) for r in records]
    for fut in tqdm(
        cf.as_completed(futures),
        total=len(futures),
        desc="Metadata/checksum audit"
    ):
        enriched.append(fut.result())

manifest = pd.DataFrame(enriched)
manifest["release_period"] = pd.PeriodIndex(manifest["release_month"], freq="M")
manifest["size_gib"] = manifest["content_length"] / (1024**3)
manifest["md5_valid_shape"] = (
    manifest["ncbi_md5"].fillna("").str.fullmatch(r"[a-f0-9]{32}")
)

manifest = manifest.sort_values(
    ["release_period", "data_model", "format_generation"]
).reset_index(drop=True)

print(
    "Files with valid-looking NCBI MD5:",
    int(manifest["md5_valid_shape"].sum()),
    "/",
    len(manifest),
)
display(manifest[[
    "release_month", "data_model", "format_generation",
    "size_gib", "http_status", "ncbi_md5", "last_modified"
]].head(20))

Metadata/checksum audit:   0%|          | 0/172 [00:00<?, ?it/s]

Files with valid-looking NCBI MD5: 172 / 172


,release_month,data_model,format_generation,size_gib,http_status,ncbi_md5,last_modified
0,2021-01,RCV,legacy,1.150045,200,aee982416ae4dc0c79e40146eefc20b0,"Fri, 03 Dec 2021 14:47:12 GMT"
1,2021-01,VCV,legacy,1.014063,200,8990a80332ff5d6fabf2b0c036d85973,"Fri, 03 Dec 2021 14:47:12 GMT"
2,2021-02,RCV,legacy,1.192228,200,5e523ad4e868cadc37fa4aab37252d86,"Fri, 03 Dec 2021 14:47:12 GMT"
3,2021-02,VCV,legacy,1.033639,200,039899d9e15319cc1619321cdafab9a4,"Fri, 03 Dec 2021 14:47:12 GMT"
4,2021-03,RCV,legacy,1.190415,200,97df93fd5053fcda8a0bec6e2e5fd92a,"Fri, 03 Dec 2021 14:47:12 GMT"
5,2021-03,VCV,legacy,1.037517,200,c6bf8c32bca40c67afa1733bbebd1b41,"Fri, 03 Dec 2021 14:47:12 GMT"
6,2021-04,RCV,legacy,1.262062,200,8521631ded8ef0da2dc8ec318a656771,"Fri, 03 Dec 2021 14:47:12 GMT"
7,2021-04,VCV,legacy,1.109461,200,6158516dc0f79c023a423e52ed95ff14,"Fri, 03 Dec 2021 14:47:12 GMT"
8,2021-05,RCV,legacy,1.287261,200,cb429b4e4dcf932466501781071f5a6e,"Fri, 03 Dec 2021 14:47:12 GMT"
9,2021-05,VCV,legacy,1.129750,200,bbf2716f8d2e44736ed6e9e3a5be6160,"Fri, 03 Dec 2021 14:47:12 GMT"


## 7. Resolve overlapping current/legacy months into a canonical source policy

For a given `(data_model, release_month)`:
- prefer the **current** XML generation if it exists;
- otherwise retain the **legacy** generation.

Nothing is discarded from the full audit manifest. The preferred source is marked separately for downstream ingestion.

In [7]:
format_priority = {"current": 0, "legacy": 1}
manifest["format_priority"] = (
    manifest["format_generation"].map(format_priority).fillna(99)
)

manifest = manifest.sort_values(
    ["data_model", "release_period", "format_priority", "url"]
).reset_index(drop=True)

manifest["canonical"] = False
first_idx = (
    manifest.groupby(["data_model", "release_month"], sort=False)
    .head(1)
    .index
)
manifest.loc[first_idx, "canonical"] = True

canonical = (
    manifest[manifest["canonical"]]
    .sort_values(["release_period", "data_model"])
    .reset_index(drop=True)
)

dupe_check = canonical.duplicated(["data_model", "release_month"]).sum()
assert dupe_check == 0, f"Canonical manifest has {dupe_check} duplicate model-month pairs."

print("Canonical monthly sources:", len(canonical))

print("\nFormat counts in canonical manifest:")
display(
    canonical.groupby(["data_model", "format_generation"])
    .size().rename("n").reset_index()
)

print("\nSchema-transition overlap examples:")
overlap = (
    manifest.groupby(["data_model", "release_month"])["format_generation"]
    .nunique()
    .reset_index(name="n_formats")
)
overlap = overlap[overlap["n_formats"] > 1]
display(overlap.tail(24))

Canonical monthly sources: 136

Format counts in canonical manifest:


,data_model,format_generation,n
0,RCV,current,31
1,RCV,legacy,37
2,VCV,current,31
3,VCV,legacy,37



Schema-transition overlap examples:


,data_model,release_month,n_formats
49,RCV,2025-02,2
50,RCV,2025-03,2
51,RCV,2025-04,2
52,RCV,2025-05,2
53,RCV,2025-06,2
54,RCV,2025-07,2
105,VCV,2024-02,2
106,VCV,2024-03,2
107,VCV,2024-04,2
108,VCV,2024-05,2


## 8. Month-coverage audit

GES 3.0 needs a multi-snapshot trajectory. Before any large download, this table determines whether each candidate month has a canonical VCV and RCV source.

Missing releases are **reported**, never silently filled.

In [8]:
min_period = pd.Period(f"{START_YEAR}-01", freq="M")
max_period = canonical["release_period"].max()

if END_YEAR is not None:
    max_period = min(max_period, pd.Period(f"{END_YEAR}-12", freq="M"))

expected = pd.period_range(min_period, max_period, freq="M")
coverage = pd.DataFrame({"release_period": expected})
coverage["release_month"] = coverage["release_period"].astype(str)

for model in ["VCV", "RCV"]:
    m = canonical[canonical["data_model"] == model][[
        "release_month", "format_generation", "url", "ncbi_md5"
    ]].copy()

    m = m.rename(columns={
        "format_generation": f"{model.lower()}_format",
        "url": f"{model.lower()}_url",
        "ncbi_md5": f"{model.lower()}_ncbi_md5",
    })
    coverage = coverage.merge(m, on="release_month", how="left")

coverage["has_vcv"] = coverage["vcv_url"].notna()
coverage["has_rcv"] = coverage["rcv_url"].notna()
coverage["paired_vcv_rcv"] = coverage["has_vcv"] & coverage["has_rcv"]

gap_rows = coverage[~coverage["paired_vcv_rcv"]].copy()

print("Candidate months:", len(coverage))
print("Paired VCV+RCV months:", int(coverage["paired_vcv_rcv"].sum()))
print("Months with any VCV/RCV gap:", len(gap_rows))

if len(gap_rows):
    display(gap_rows)
else:
    print("No paired-month gaps detected in the candidate interval.")

display(coverage.tail(24))

Candidate months: 68
Paired VCV+RCV months: 68
Months with any VCV/RCV gap: 0
No paired-month gaps detected in the candidate interval.


,release_period,release_month,vcv_format,vcv_url,vcv_ncbi_md5,rcv_format,rcv_url,rcv_ncbi_md5,has_vcv,has_rcv,paired_vcv_rcv
44,2024-09,2024-09,current,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/a...,bdb0ad8f4a79ce2923d0c0bd8a3c2293,current,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,46c415cbeb91f689443954bcf9697f9d,True,True,True
45,2024-10,2024-10,current,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/a...,21cf8ef346e309a5f66496f1aacca5eb,current,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,ad8e78d3bc55f09de23d63da91fd5620,True,True,True
46,2024-11,2024-11,current,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/a...,2cea4febe1370a97d504d4613f6d2ec7,current,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,92805f60bab1fde52386cf6e9c74fd4c,True,True,True
47,2024-12,2024-12,current,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/a...,eac32e9f15fe211f815d15b934146f35,current,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,bb25fa6815874795b3488afd52d75842,True,True,True
48,2025-01,2025-01,current,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/C...,48a4b1e84ff921916f97696c45fb9343,current,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,41f3741c1c2fa386393466c21251a4f7,True,True,True
49,2025-02,2025-02,current,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/C...,9ab805f0abb0b72099bc90eb9474fa22,current,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,63ad60883dd4732bb77e047fa6b437f8,True,True,True
50,2025-03,2025-03,current,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/C...,aef890b78cf3eea313ed2fd1775371f0,current,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,396e67a3a25b3f97d638285230a510ce,True,True,True
51,2025-04,2025-04,current,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/C...,435204cd071f94a4f2015e7672847c54,current,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,778e8e5cf9eb35fc64efa6a10c3392d9,True,True,True
52,2025-05,2025-05,current,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/C...,99e4f1c1cbcecf33196311cf227bc421,current,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,a29f4a12f45a460b7a1727ed796b79e8,True,True,True
53,2025-06,2025-06,current,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/C...,678575cef6d4a3b274db6053937e4588,current,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,9d6d0d8fed411f9e5b0d4de3da68c16e,True,True,True


## 9. Schema/XSD catalog

This inventories the public XSD families that the Stage 01 parser must bridge:
- current VCV;
- current RCV;
- legacy VCV;
- legacy RCV.

In [9]:
SCHEMA_SOURCES = pd.DataFrame([
    {
        "data_model": "VCV",
        "format_generation": "current",
        "xsd_url": "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xsd_public/",
    },
    {
        "data_model": "RCV",
        "format_generation": "current",
        "xsd_url": "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xsd_public/RCV/",
    },
    {
        "data_model": "VCV",
        "format_generation": "legacy",
        "xsd_url": "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xsd_public/VCV_xsd_old_format/",
    },
    {
        "data_model": "RCV",
        "format_generation": "legacy",
        "xsd_url": "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xsd_public/RCV_xsd_old_format/",
    },
])

schema_rows = []

for _, s in SCHEMA_SOURCES.iterrows():
    print("Scanning:", s["xsd_url"])
    try:
        idx = list_apache_links(s["xsd_url"])
    except Exception as e:
        schema_rows.append({
            **s.to_dict(),
            "name": None,
            "url": None,
            "scan_error": repr(e),
        })
        continue

    keep = idx[
        idx["name"].str.lower().str.endswith(
            (".xsd", ".xml", ".txt", ".md5"), na=False
        )
    ].copy()

    if keep.empty:
        schema_rows.append({
            **s.to_dict(),
            "name": None,
            "url": None,
            "scan_error": "No matching schema/support files found.",
        })
    else:
        for _, r in keep.iterrows():
            schema_rows.append({
                **s.to_dict(),
                "name": r["name"],
                "url": r["url"],
                "scan_error": None,
            })

schema_catalog = pd.DataFrame(schema_rows)

if not schema_catalog.empty:
    schema_catalog["is_xsd"] = (
        schema_catalog["name"].fillna("").str.lower().str.endswith(".xsd")
    )
    xsd_indices = schema_catalog.index[schema_catalog["is_xsd"]].tolist()

    def schema_head(i):
        return i, safe_head(schema_catalog.loc[i, "url"])

    with cf.ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = [ex.submit(schema_head, i) for i in xsd_indices]
        for fut in cf.as_completed(futures):
            i, h = fut.result()
            for k, v in h.items():
                schema_catalog.loc[i, k] = v

display(schema_catalog)

Scanning: https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xsd_public/
Scanning: https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xsd_public/RCV/
Scanning: https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xsd_public/VCV_xsd_old_format/
Scanning: https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xsd_public/RCV_xsd_old_format/


,data_model,format_generation,xsd_url,name,url,scan_error,is_xsd,http_status,content_length,last_modified,etag,accept_ranges,head_error
0,VCV,current,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xsd_p...,ClinVar_VCV.xsd,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xsd_p...,None,True,200.0,15974.0,"Fri, 27 Feb 2026 04:02:46 GMT",None,bytes,None
1,VCV,current,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xsd_p...,ClinVar_VCV.xsd.md5,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xsd_p...,None,False,NaN,NaN,NaN,NaN,NaN,NaN
2,VCV,current,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xsd_p...,ClinVar_VCV_2.0.xsd,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xsd_p...,None,True,200.0,15258.0,"Mon, 29 Jan 2024 18:30:08 GMT",None,bytes,None
3,VCV,current,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xsd_p...,ClinVar_VCV_2.0.xsd.md5,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xsd_p...,None,False,NaN,NaN,NaN,NaN,NaN,NaN
4,VCV,current,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xsd_p...,ClinVar_VCV_2.1.xsd,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xsd_p...,None,True,200.0,150630.0,"Mon, 09 Dec 2024 00:27:55 GMT",None,bytes,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
178,RCV,legacy,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xsd_p...,clinvar_public_1.73.xsd,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xsd_p...,None,True,200.0,10307.0,"Tue, 15 Apr 2025 17:00:14 GMT",None,bytes,None
179,RCV,legacy,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xsd_p...,clinvar_public_1.8.xsd,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xsd_p...,None,True,200.0,57380.0,"Mon, 07 Jul 2014 12:47:26 GMT",None,bytes,None
180,RCV,legacy,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xsd_p...,clinvar_public_1.9.xsd,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xsd_p...,None,True,200.0,7857.0,"Fri, 08 Aug 2014 20:55:16 GMT",None,bytes,None
181,RCV,legacy,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xsd_p...,clinvar_public_weekly.xsd,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xsd_p...,None,True,200.0,10307.0,"Tue, 15 Apr 2025 17:00:14 GMT",None,bytes,None


## 10. Explicit schema-bridge plan

The bridge plan prevents a major methodological error: treating historical and current classification structures as identical.

The Stage 01 parser should dispatch on both `data_model` and `format_generation`, and preserve a classification-axis field so germline and somatic records cannot be accidentally pooled.

In [10]:
schema_bridge_plan = pd.DataFrame([
    {
        "data_model": "VCV",
        "format_generation": "current",
        "downstream_role": "Variant identity plus nested submitted assertions",
        "classification_handling": (
            "Keep germline and somatic classification axes separate; "
            "primary GES 3.0 model uses germline."
        ),
        "parser_strategy": "Current VCV XSD-driven parser.",
        "canonical_preference": 1,
    },
    {
        "data_model": "RCV",
        "format_generation": "current",
        "downstream_role": (
            "PRIMARY variant-condition monthly prediction state plus nested SCVs"
        ),
        "classification_handling": (
            "Keep germline and somatic classification axes separate; "
            "primary GES 3.0 model uses germline."
        ),
        "parser_strategy": "Current RCV XSD-driven parser.",
        "canonical_preference": 1,
    },
    {
        "data_model": "VCV",
        "format_generation": "legacy",
        "downstream_role": (
            "Historical variant identity where current-format month is unavailable"
        ),
        "classification_handling": (
            "Map the legacy single-classification representation into the "
            "normalized axis with explicit provenance; never guess missing modern semantics."
        ),
        "parser_strategy": "Legacy VCV parser -> normalized bridge schema.",
        "canonical_preference": 2,
    },
    {
        "data_model": "RCV",
        "format_generation": "legacy",
        "downstream_role": (
            "Historical variant-condition state where current-format month is unavailable"
        ),
        "classification_handling": (
            "Map the legacy single-classification representation into the "
            "normalized axis with explicit provenance; never guess missing modern semantics."
        ),
        "parser_strategy": "Legacy RCV parser -> normalized bridge schema.",
        "canonical_preference": 2,
    },
])

display(schema_bridge_plan)

,data_model,format_generation,downstream_role,classification_handling,parser_strategy,canonical_preference
0,VCV,current,Variant identity plus nested submitted assertions,Keep germline and somatic classification axes ...,Current VCV XSD-driven parser.,1
1,RCV,current,PRIMARY variant-condition monthly prediction s...,Keep germline and somatic classification axes ...,Current RCV XSD-driven parser.,1
2,VCV,legacy,Historical variant identity where current-form...,Map the legacy single-classification represent...,Legacy VCV parser -> normalized bridge schema.,2
3,RCV,legacy,Historical variant-condition state where curre...,Map the legacy single-classification represent...,Legacy RCV parser -> normalized bridge schema.,2


## 11. Optional lightweight current-format sample probe

NCBI publishes a small VCV sample XML. This sanity check records the root, namespaces, observed element names, and SHA-256 of the sample bytes.

It is **not** a substitute for validating each downloaded monthly release in Stage 01.

In [11]:
from lxml import etree

SAMPLE_VCV_URL = (
    "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/sample_xml/"
    "VCV_XML_VCV000091629.xml"
)

sample_probe = {}

try:
    r = SESSION.get(SAMPLE_VCV_URL, timeout=REQUEST_TIMEOUT)
    r.raise_for_status()
    raw = r.content
    root = etree.fromstring(raw)

    q = etree.QName(root)
    tags = []

    for el in root.iter():
        try:
            tags.append(etree.QName(el).localname)
        except Exception:
            pass

    sample_probe = {
        "url": SAMPLE_VCV_URL,
        "bytes": len(raw),
        "sha256": hashlib.sha256(raw).hexdigest(),
        "root_localname": q.localname,
        "root_namespace": q.namespace,
        "unique_element_count": len(set(tags)),
        "first_40_unique_elements": list(dict.fromkeys(tags))[:40],
    }

    print(json.dumps(sample_probe, indent=2))
except Exception as e:
    sample_probe = {"url": SAMPLE_VCV_URL, "error": repr(e)}
    print("Sample probe warning:", repr(e))

{
  "url": "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/sample_xml/VCV_XML_VCV000091629.xml",
  "bytes": 193155,
  "sha256": "028c83d8bb66f9a39cb9fc044a57968c901fd351075ccea004c8ce147a57cef0",
  "root_localname": "ClinVarResult-Set",
  "root_namespace": null,
  "unique_element_count": 73,
  "first_40_unique_elements": [
    "ClinVarResult-Set",
    "VariationArchive",
    "RecordStatus",
    "Species",
    "ClassifiedRecord",
    "SimpleAllele",
    "GeneList",
    "Gene",
    "Location",
    "CytogeneticLocation",
    "SequenceLocation",
    "OMIM",
    "Haploinsufficiency",
    "Triplosensitivity",
    "Property",
    "Name",
    "CanonicalSPDI",
    "VariantType",
    "OtherNameList",
    "HGVSlist",
    "HGVS",
    "NucleotideExpression",
    "Expression",
    "MolecularConsequence",
    "XRefList",
    "XRef",
    "RCVList",
    "RCVAccession",
    "ClassifiedConditionList",
    "ClassifiedCondition",
    "RCVClassifications",
    "GermlineClassification",
    "ReviewStatus",
   

## 12. Automated Stage 00 QC invariants

These checks validate source selection and manifest integrity. They do **not** invent a scientific success threshold for archive coverage.

If a month is missing, it remains an explicit gap for review.

In [12]:
qc = {}

qc["canonical_unique_model_month"] = not canonical.duplicated(
    ["data_model", "release_month"]
).any()

qc["no_latest_symlink_in_canonical"] = not canonical["name"].str.contains(
    "00-latest", regex=False
).any()

qc["release_month_matches_filename"] = (
    canonical["release_month"] == canonical["name"].map(parse_release_month)
).all()

qc["all_urls_https"] = canonical["url"].str.startswith("https://").all()

observed_status = canonical["http_status"].dropna().astype(int)
qc["all_http_status_ok_where_observed"] = (
    observed_status.between(200, 399).all() if len(observed_status) else True
)

present_md5 = canonical.loc[canonical["ncbi_md5"].notna(), "ncbi_md5"]
qc["md5_shape_ok_where_present"] = (
    present_md5.str.fullmatch(r"[a-f0-9]{32}").all()
    if len(present_md5)
    else True
)

available_current = (
    manifest[manifest["format_generation"] == "current"][
        ["data_model", "release_month"]
    ]
    .drop_duplicates()
    .assign(current_available=True)
)

canonical_pref = canonical.merge(
    available_current,
    on=["data_model", "release_month"],
    how="left",
)

bad_pref = canonical_pref[
    canonical_pref["current_available"].fillna(False)
    & (canonical_pref["format_generation"] != "current")
]

qc["current_preferred_when_available"] = len(bad_pref) == 0

for k, v in qc.items():
    print(f"{'PASS' if v else 'FAIL'} — {k}")

if not all(qc.values()):
    failed = [k for k, v in qc.items() if not v]
    raise AssertionError("Stage 00 QC failed: " + ", ".join(failed))

print("\nAll structural Stage 00 QC invariants passed.")

PASS — canonical_unique_model_month
PASS — no_latest_symlink_in_canonical
PASS — release_month_matches_filename
PASS — all_urls_https
PASS — all_http_status_ok_where_observed
PASS — md5_shape_ok_where_present
PASS — current_preferred_when_available

All structural Stage 00 QC invariants passed.


/tmp/ipykernel_1489/3621553501.py:44: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  canonical_pref["current_available"].fillna(False)


## 13. Freeze Stage 00 audit artifacts

This is an **audit freeze**, not the final GES 3.0 cohort freeze.

Stage 01 should consume `clinvar_release_manifest_canonical.csv` rather than rediscovering URLs ad hoc.

In [13]:
manifest_out = manifest.sort_values(
    ["release_period", "data_model", "format_priority", "url"]
).copy()

canonical_out = canonical.sort_values(
    ["release_period", "data_model"]
).copy()

coverage_out = coverage.sort_values(
    "release_period"
).copy()

schema_catalog_out = schema_catalog.sort_values(
    ["data_model", "format_generation", "name"],
    na_position="last",
).copy()

for df in [manifest_out, canonical_out, coverage_out]:
    if "release_period" in df.columns:
        df["release_period"] = df["release_period"].astype(str)

manifest_path = OUTDIR / "clinvar_release_manifest_all.csv"
canonical_path = OUTDIR / "clinvar_release_manifest_canonical.csv"
coverage_path = OUTDIR / "clinvar_month_coverage.csv"
schema_path = OUTDIR / "clinvar_schema_catalog.csv"
bridge_path = OUTDIR / "clinvar_schema_bridge_plan.csv"

manifest_out.to_csv(manifest_path, index=False)
canonical_out.to_csv(canonical_path, index=False)
coverage_out.to_csv(coverage_path, index=False)
schema_catalog_out.to_csv(schema_path, index=False)
schema_bridge_plan.to_csv(bridge_path, index=False)

runtime_metadata = {
    "run_utc": RUN_UTC,
    "python": sys.version,
    "platform": platform.platform(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "requests": requests.__version__,
    "beautifulsoup4": bs4.__version__,
    "lxml": lxml.__version__,
    "candidate_start_year": START_YEAR,
    "candidate_end_year": END_YEAR,
    "allow_full_xml_download": ALLOW_FULL_XML_DOWNLOAD,
}

summary = {
    "project": "GES 3.0",
    "stage": "00_archive_manifest_and_schema_audit",
    "run_utc": RUN_UTC,
    "candidate_start_month": str(min_period),
    "latest_discovered_canonical_month": str(max_period),
    "n_all_discovered_files": int(len(manifest_out)),
    "n_canonical_files": int(len(canonical_out)),
    "n_candidate_months": int(len(coverage_out)),
    "n_paired_vcv_rcv_months": int(coverage_out["paired_vcv_rcv"].sum()),
    "n_gap_months": int((~coverage_out["paired_vcv_rcv"]).sum()),
    "canonical_format_counts": (
        canonical.groupby(["data_model", "format_generation"])
        .size()
        .rename("n")
        .reset_index()
        .to_dict("records")
    ),
    "qc": {k: bool(v) for k, v in qc.items()},
    "sample_probe": sample_probe,
    "design_status": (
        "Archive/schema audit complete. Final study window remains unfrozen "
        "pending coverage review, Stage 01 normalization, linkage QC, "
        "and event-prevalence analysis."
    ),
}

(OUTDIR / "stage00_runtime_metadata.json").write_text(
    json.dumps(runtime_metadata, indent=2),
    encoding="utf-8",
)

(OUTDIR / "stage00_audit_summary.json").write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8",
)

artifact_hashes = []

for p in sorted(OUTDIR.glob("*")):
    if p.is_file() and p.suffix in {".csv", ".json"}:
        artifact_hashes.append({
            "file": p.name,
            "bytes": p.stat().st_size,
            "sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
        })

hash_df = pd.DataFrame(artifact_hashes)
hash_df.to_csv(OUTDIR / "stage00_artifact_sha256.csv", index=False)

print(json.dumps(summary, indent=2))

print("\nFrozen Stage 00 files:")
for p in sorted(OUTDIR.iterdir()):
    if p.is_file():
        print(" -", p.name)

{
  "project": "GES 3.0",
  "stage": "00_archive_manifest_and_schema_audit",
  "run_utc": "2026-08-16T17:11:48.502727+00:00",
  "candidate_start_month": "2021-01",
  "latest_discovered_canonical_month": "2026-08",
  "n_all_discovered_files": 172,
  "n_canonical_files": 136,
  "n_candidate_months": 68,
  "n_paired_vcv_rcv_months": 68,
  "n_gap_months": 0,
  "canonical_format_counts": [
    {
      "data_model": "RCV",
      "format_generation": "current",
      "n": 31
    },
    {
      "data_model": "RCV",
      "format_generation": "legacy",
      "n": 37
    },
    {
      "data_model": "VCV",
      "format_generation": "current",
      "n": 31
    },
    {
      "data_model": "VCV",
      "format_generation": "legacy",
      "n": 37
    }
  ],
  "qc": {
    "canonical_unique_model_month": true,
    "no_latest_symlink_in_canonical": true,
    "release_month_matches_filename": true,
    "all_urls_https": true,
    "all_http_status_ok_where_observed": true,
    "md5_shape_ok_where_pre

## 14. Build a single artifact bundle

In [14]:
import shutil

zip_base = "/content/GES3_STAGE00_ARTIFACTS"
zip_path = shutil.make_archive(zip_base, "zip", root_dir=OUTDIR)

print("Artifact bundle:", zip_path)
print("Bytes:", Path(zip_path).stat().st_size)

Artifact bundle: /content/GES3_STAGE00_ARTIFACTS.zip
Bytes: 33927


## 15. Interpretation and next notebook

### What this notebook establishes
After a successful run, GES 3.0 has a reproducible answer to:
- Which monthly ClinVar releases exist in the candidate historical window?
- Which are VCV versus RCV?
- Which use current versus legacy XML?
- Which NCBI checksum belongs to each release?
- Where does current/legacy overlap occur?
- Which month-source should Stage 01 treat as canonical?
- Are there missing paired VCV/RCV months?
- Which public XSD families must the parser bridge?

### What this notebook does **not** claim
It does not claim stable cross-release RCV/VCV linkage, event prevalence, or forecasting feasibility yet.

### Next notebook
**GES 3.0 — Notebook 01: `clinvar_xml_normalizer`**

Stage 01 should:
1. consume the frozen canonical manifest from this notebook;
2. download selected monthly releases one at a time;
3. verify NCBI MD5 and compute local SHA-256;
4. parse VCV / RCV / nested SCV content with explicit current-vs-legacy dispatch;
5. normalize germline classification semantics;
6. write immutable release-partitioned Parquet tables;
7. preserve source-release lineage and schema fingerprints;
8. emit parser QC and row-count reports.

**Do not move to modeling before Stage 01–06 establish a leakage-safe longitudinal evidence state system.**